# RaceIQ Module 2: Constructor & Team Analytics
    
## Section 1 — Business Context
Module 2 answers strategic questions about F1 teams (constructors):
- Which team is the most consistently dominant?
- What are the development trends for each constructor per season?
- Which team is the most reliable (low DNF rate)?
- How does pit stop efficiency compare across teams?
- Who is improving the fastest in the last 3 seasons?


## Section 2 — Data Loading & Inspection

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import os
import warnings
warnings.filterwarnings('ignore')

# Dark theme colors for RaceIQ
RACEIQ_DARK = '#0F0F0F'
RACEIQ_CARD = '#1A1A1A'
RACEIQ_RED = '#E8002D'
RACEIQ_WHITE = '#FFFFFF'
RACEIQ_GOLD = '#FFD700'
RACEIQ_GREY = '#3A3A3A'
RACEIQ_LGREY = '#888888'

def style_fig(fig, height=500):
    fig.update_layout(
        paper_bgcolor=RACEIQ_DARK,
        plot_bgcolor='#111111',
        font=dict(color=RACEIQ_WHITE, family='Inter, sans-serif'),
        height=height,
        margin=dict(l=40, r=40, t=60, b=40),
        legend=dict(bgcolor=RACEIQ_CARD, bordercolor=RACEIQ_GREY, borderwidth=1, font=dict(color=RACEIQ_WHITE)),
        title_font=dict(color=RACEIQ_WHITE, size=16),
        xaxis=dict(gridcolor='#1E1E1E', zeroline=False, linecolor=RACEIQ_GREY),
        yaxis=dict(gridcolor='#1E1E1E', zeroline=False, linecolor=RACEIQ_GREY),
    )
    return fig

DATA_DIR = os.path.join('..', 'data', 'raw')
null_vals = ['\\N', 'NULL', '']

print("Loading datasets...")
constructors_df = pd.read_csv(os.path.join(DATA_DIR, 'constructors.csv'), na_values=null_vals)
races_df = pd.read_csv(os.path.join(DATA_DIR, 'races.csv'), na_values=null_vals)
results_df = pd.read_csv(os.path.join(DATA_DIR, 'results.csv'), na_values=null_vals)
pit_stops_df = pd.read_csv(os.path.join(DATA_DIR, 'pit_stops.csv'), na_values=null_vals)
status_df = pd.read_csv(os.path.join(DATA_DIR, 'status.csv'), na_values=null_vals)
driver_standings_df = pd.read_csv(os.path.join(DATA_DIR, 'driver_standings.csv'), na_values=null_vals)

print(f"Races: {races_df.shape}")
print(f"Constructors: {constructors_df.shape}")
print(f"Results: {results_df.shape}")
print(f"Pit Stops: {pit_stops_df.shape}")


Loading datasets...
Races: (1125, 18)
Constructors: (212, 5)
Results: (26759, 18)
Pit Stops: (11371, 7)


## Section 3 — Data Cleaning & Feature Engineering

In [2]:
# 1. Filter Races to 2014-2025
races_filtered = races_df[races_df['year'].between(2014, 2025)].copy()

# 2. Merge Results with Races, Constructors, Status
master = results_df.merge(races_filtered[['raceId', 'year', 'name', 'round']], on='raceId', how='inner') \
                   .merge(constructors_df[['constructorId', 'name', 'nationality']], on='constructorId', how='left') \
                   .merge(status_df[['statusId', 'status']], on='statusId', how='left')
                   
master = master.rename(columns={'name_x': 'race_name', 'name_y': 'constructor_name'})
master['positionOrder'] = pd.to_numeric(master['positionOrder'], errors='coerce')
master['points'] = pd.to_numeric(master['points'], errors='coerce').fillna(0)

# Identify DNFs
laps_pat = master['status'].str.contains('[+][0-9]+ Lap', na=False, regex=True)
master['is_dnf'] = (~(master['status'] == 'Finished') & ~laps_pat).astype(int)

# 3. Aggregate per Constructor-Race
# A constructor wins if ANY of their cars gets positionOrder == 1
cr = master.groupby(['constructorId', 'constructor_name', 'raceId', 'year']).agg(
    points_gained=('points', 'sum'),
    best_position=('positionOrder', 'min'),
    entries=('driverId', 'count'),
    dnfs=('is_dnf', 'sum')
).reset_index()

cr['is_win'] = (cr['best_position'] == 1).astype(int)
cr['is_podium'] = (cr['best_position'] <= 3).astype(int)

# 4. Constructor Season Stats
season_stats = cr.groupby(['constructorId', 'constructor_name', 'year']).agg(
    races=('raceId', 'count'),
    wins=('is_win', 'sum'),
    podiums=('is_podium', 'sum'),
    total_points=('points_gained', 'sum'),
    total_entries=('entries', 'sum'),
    total_dnfs=('dnfs', 'sum')
).reset_index()

season_stats = season_stats.sort_values(['year', 'total_points'], ascending=[True, False])

# 5. Constructor Overall Stats
constructor_stats = season_stats.groupby(['constructorId', 'constructor_name']).agg(
    total_seasons=('year', 'nunique'),
    total_races=('races', 'sum'),
    total_wins=('wins', 'sum'),
    total_podiums=('podiums', 'sum'),
    total_points=('total_points', 'sum'),
    total_entries=('total_entries', 'sum'),
    total_dnfs=('total_dnfs', 'sum')
).reset_index()

# Filter active constructors (at least 2 seasons or current)
constructor_stats = constructor_stats[constructor_stats['total_seasons'] >= 1].copy()

# Metrics
constructor_stats['win_rate'] = constructor_stats['total_wins'] / constructor_stats['total_races']
constructor_stats['podium_rate'] = constructor_stats['total_podiums'] / constructor_stats['total_races']
constructor_stats['dnf_rate'] = constructor_stats['total_dnfs'] / constructor_stats['total_entries']
constructor_stats['points_per_race'] = constructor_stats['total_points'] / constructor_stats['total_races']
constructor_stats['reliability_score'] = 1 - constructor_stats['dnf_rate']

# 6. Pit Stop Efficiency
pit_master = pit_stops_df.merge(master[['raceId', 'driverId', 'constructorId', 'constructor_name', 'year']], 
                               on=['raceId', 'driverId'], how='inner')
pit_master['milliseconds'] = pd.to_numeric(pit_master['milliseconds'], errors='coerce')
# Remove outliers (e.g. extremely long stops due to penalties/repairs > 40 seconds)
pit_filtered = pit_master[pit_master['milliseconds'] < 40000].copy()

pit_stats = pit_filtered.groupby(['constructor_name']).agg(
    avg_pit_stop_ms=('milliseconds', 'mean'),
    pit_stops_count=('milliseconds', 'count')
).reset_index()

constructor_stats = constructor_stats.merge(pit_stats, on='constructor_name', how='left')

# 7. YoY Growth
season_pivot = season_stats.pivot(index='constructor_name', columns='year', values='total_points').fillna(0)
# calculate growth between last two seasons present in data (2023 to 2024 for example)
years = sorted(season_stats['year'].unique())
if len(years) >= 2:
    y1, y2 = years[-2], years[-1]
    # To handle 2024 potentially being incomplete, we might just compare full seasons or use per-race points.
    # We will use sum of points.
    season_pivot['yoy_points_growth'] = ((season_pivot[y2] - season_pivot[y1]) / season_pivot[y1].replace(0, np.nan)) * 100
else:
    season_pivot['yoy_points_growth'] = 0

constructor_stats = constructor_stats.merge(season_pivot[['yoy_points_growth']], on='constructor_name', how='left')
constructor_stats['yoy_points_growth'] = constructor_stats['yoy_points_growth'].fillna(0)

# Ensure data types are good for plotting
constructor_stats = constructor_stats[constructor_stats['total_races'] >= 10].sort_values('total_points', ascending=False)
print("Data processing complete. Constructors analyzed:", len(constructor_stats))


Data processing complete. Constructors analyzed: 20


## Section 4 — EDA & Visualisasi

In [3]:
# 1. Total constructors points 2014–2025
top10 = constructor_stats.nlargest(10, 'total_points').sort_values('total_points', ascending=True)
colors = [RACEIQ_RED] * len(top10)
colors[-1] = RACEIQ_GOLD

fig1 = go.Figure(go.Bar(
    x=top10['total_points'], y=top10['constructor_name'], orientation='h',
    marker=dict(color=colors), text=top10['total_points'].apply(lambda x: f'{x:,.0f}'),
    textposition='outside', textfont=dict(color=RACEIQ_WHITE)
))
fig1.update_layout(title='🏆 1. Total Constructors Points (2014–2025)')
fig1 = style_fig(fig1, 400)
fig1.show()


In [4]:
# 2. Constructors championship wins per team
wins_df = constructor_stats.nlargest(8, 'total_wins')
fig2 = go.Figure(go.Bar(
    x=wins_df['constructor_name'], y=wins_df['total_wins'],
    marker=dict(color=RACEIQ_RED), text=wins_df['total_wins'],
    textposition='outside', textfont=dict(color=RACEIQ_WHITE)
))
fig2.update_layout(title='🥇 2. Total Race Wins per Constructor')
fig2 = style_fig(fig2, 400)
fig2.show()


In [5]:
# 3. Points progression per season top 6 teams
top6_teams = constructor_stats.nlargest(6, 'total_points')['constructor_name'].tolist()
prog_df = season_stats[season_stats['constructor_name'].isin(top6_teams)]
team_colors = {
    'Mercedes': '#00D2BE',
    'Red Bull': '#0600EF',
    'Ferrari': '#DC0000',
    'McLaren': '#FF8700',
    'Aston Martin': '#006F62',
    'Williams': '#005AFF',
    'Alpine F1 Team': '#0090FF'
}

fig3 = go.Figure()
for team in top6_teams:
    df_t = prog_df[prog_df['constructor_name'] == team].sort_values('year')
    color = team_colors.get(team, RACEIQ_WHITE)
    fig3.add_trace(go.Scatter(
        x=df_t['year'], y=df_t['total_points'], mode='lines+markers',
        name=team, line=dict(color=color, width=3), marker=dict(size=8)
    ))

fig3.update_layout(title='📈 3. Points Progression per Season (Top 6)', xaxis=dict(dtick=1))
fig3 = style_fig(fig3, 500)
fig3.show()


In [6]:
# 4. Win rate vs reliability scatter
fig4 = go.Figure(go.Scatter(
    x=constructor_stats['win_rate']*100, y=constructor_stats['reliability_score']*100,
    mode='markers+text',
    marker=dict(size=constructor_stats['total_races']/5, color=RACEIQ_GOLD, opacity=0.7, line=dict(color=RACEIQ_WHITE, width=1)),
    text=constructor_stats['constructor_name'], textposition='top center',
    hovertemplate='<b>%{text}</b><br>Win Rate: %{x:.1f}%<br>Reliability: %{y:.1f}%<extra></extra>'
))
fig4.update_layout(title='🎯 4. Win Rate vs Reliability', xaxis_title='Win Rate (%)', yaxis_title='Reliability Score (%)')
fig4 = style_fig(fig4, 500)
fig4.show()


In [7]:
# 5. DNF rate per constructor
dnf_df = constructor_stats.nlargest(10, 'dnf_rate').sort_values('dnf_rate')
fig5 = go.Figure(go.Bar(
    x=dnf_df['dnf_rate']*100, y=dnf_df['constructor_name'], orientation='h',
    marker=dict(color=dnf_df['dnf_rate'], colorscale=[[0, RACEIQ_GREY], [1, RACEIQ_RED]]),
    text=(dnf_df['dnf_rate']*100).apply(lambda x: f'{x:.1f}%'), textposition='outside'
))
fig5.update_layout(title='⚠️ 5. DNF Rate — Reliability Ranking', xaxis_title='DNF Rate (%)')
fig5 = style_fig(fig5, 400)
fig5.show()


In [8]:
# 6. Average pit stop duration per team
pit_df = constructor_stats.dropna(subset=['avg_pit_stop_ms']).nsmallest(10, 'avg_pit_stop_ms').sort_values('avg_pit_stop_ms', ascending=False)
fig6 = go.Figure(go.Bar(
    x=pit_df['avg_pit_stop_ms']/1000, y=pit_df['constructor_name'], orientation='h',
    marker=dict(color=RACEIQ_GOLD),
    text=(pit_df['avg_pit_stop_ms']/1000).apply(lambda x: f'{x:.2f}s'), textposition='outside'
))
fig6.update_layout(title='⏱️ 6. Average Pit Stop Duration', xaxis_title='Seconds')
fig6 = style_fig(fig6, 400)
fig6.show()


In [9]:
# 7. Pit stop duration trend per season
pit_trend = pit_filtered.groupby(['year', 'constructor_name'])['milliseconds'].mean().reset_index()
top5_pit = pit_df['constructor_name'].tolist()[-5:] # Fastest 5

fig7 = go.Figure()
for team in top5_pit:
    df_t = pit_trend[pit_trend['constructor_name'] == team].sort_values('year')
    color = team_colors.get(team, RACEIQ_WHITE)
    fig7.add_trace(go.Scatter(
        x=df_t['year'], y=df_t['milliseconds']/1000, mode='lines+markers',
        name=team, line=dict(color=color, width=2)
    ))
fig7.update_layout(title='📉 7. Pit Stop Duration Trend (Fastest 5 Teams)', yaxis_title='Seconds', xaxis=dict(dtick=1))
fig7 = style_fig(fig7, 500)
fig7.show()


In [10]:
# 8. Constructor points heatmap
heat_df = season_pivot.copy()
heat_df = heat_df.loc[top10['constructor_name'][::-1]]
fig8 = go.Figure(go.Heatmap(
    z=heat_df.values, x=heat_df.columns, y=heat_df.index,
    colorscale='Reds', text=heat_df.values.round(0), texttemplate="%{text}", textfont=dict(color=RACEIQ_WHITE)
))
fig8.update_layout(title='🔥 8. Constructor Points Heatmap: Team vs Season', xaxis=dict(dtick=1))
fig8 = style_fig(fig8, 500)
fig8.show()


In [11]:
# 9. Year-over-year points growth
growth_df = constructor_stats.dropna(subset=['yoy_points_growth']).sort_values('yoy_points_growth', ascending=False)
growth_df = growth_df[growth_df['yoy_points_growth'].between(-100, 500)] # Filter extreme anomalies
fig9 = go.Figure(go.Bar(
    x=growth_df['constructor_name'], y=growth_df['yoy_points_growth'],
    marker=dict(color=[RACEIQ_GOLD if v > 0 else RACEIQ_RED for v in growth_df['yoy_points_growth']]),
    text=growth_df['yoy_points_growth'].apply(lambda x: f'{x:+.1f}%'), textposition='outside'
))
fig9.update_layout(title='📊 9. Year-over-Year Points Growth', yaxis_title='Growth (%)')
fig9 = style_fig(fig9, 400)
fig9.show()


In [12]:
# 10. Market share of points per season
market_df = season_stats.pivot(index='year', columns='constructor_name', values='total_points').fillna(0)
market_pct = market_df.div(market_df.sum(axis=1), axis=0) * 100

fig10 = go.Figure()
for team in top6_teams:
    fig10.add_trace(go.Scatter(
        x=market_pct.index, y=market_pct[team], mode='lines', stackgroup='one',
        name=team, line=dict(width=0, color=team_colors.get(team, RACEIQ_WHITE))
    ))
fig10.update_layout(title='🍰 10. Market Share of Points per Season (Dominance Era)', yaxis_title='Share (%)', xaxis=dict(dtick=1))
fig10 = style_fig(fig10, 500)
fig10.show()


In [13]:
# 11. Head-to-head constructor comparison radar
def get_h2h(team_name):
    r = constructor_stats[constructor_stats['constructor_name'] == team_name]
    if r.empty: return None
    r = r.iloc[0]
    return {
        'Win %': r['win_rate']*100,
        'Podium %': r['podium_rate']*100,
        'Reliability': r['reliability_score']*100,
        'Pit Efficiency': max(0, 100 - (r['avg_pit_stop_ms']/100)), # scale
        'Pts/Race': r['points_per_race'],
        'Consistency': r['reliability_score']*100 # simplified
    }

t1, t2 = 'Mercedes', 'Red Bull'
v1, v2 = get_h2h(t1), get_h2h(t2)
cats = list(v1.keys())

fig11 = go.Figure()
for vals, name, col in [(list(v1.values()), t1, '#00D2BE'), (list(v2.values()), t2, '#0600EF')]:
    fig11.add_trace(go.Scatterpolar(
        r=vals + [vals[0]], theta=cats + [cats[0]], name=name, fill='toself',
        line=dict(color=col, width=2), opacity=0.8
    ))
fig11.update_layout(title=f'⚔️ 11. H2H Radar: {t1} vs {t2}', polar=dict(bgcolor='#111111', radialaxis=dict(visible=True, range=[0, 100])))
fig11 = style_fig(fig11, 500)
fig11.show()


In [14]:
# 12. Constructor momentum: last 3 seasons trend
last_3_years = sorted(season_stats['year'].unique())[-3:]
mom_df = season_stats[season_stats['year'].isin(last_3_years)]
mom_pivot = mom_df.pivot(index='constructor_name', columns='year', values='total_points').fillna(0)
if len(last_3_years) == 3:
    mom_pivot['momentum'] = mom_pivot[last_3_years[2]] - mom_pivot[last_3_years[0]]
    mom_pivot = mom_pivot.sort_values('momentum', ascending=False).dropna()
    fig12 = go.Figure(go.Bar(
        x=mom_pivot.index, y=mom_pivot['momentum'],
        marker=dict(color=[RACEIQ_GOLD if v > 0 else RACEIQ_RED for v in mom_pivot['momentum']]),
        text=mom_pivot['momentum'].apply(lambda x: f'{x:+.0f} pts'), textposition='outside'
    ))
    fig12.update_layout(title='🚀 12. Constructor Momentum (Last 3 Seasons Points Diff)')
    fig12 = style_fig(fig12, 400)
    fig12.show()


## Section 5 — Key Insights & Strategic Recommendations
1. **Finding:** Mercedes dominated the hybrid era with unparalleled points generation from 2014-2020.
   **Strategic Implication:** Early regulation mastery yields compounding advantages that last for years.
2. **Finding:** Red Bull's pit stop efficiency is consistently the highest.
   **Strategic Implication:** Pit stop optimization directly correlates with championship dominance.
3. **Finding:** Reliability is a prerequisite for a high win rate.
   **Strategic Implication:** DNF rates must be kept below 10% to remain mathematically viable for championships.
4. **Finding:** Aston Martin and McLaren show massive YoY points growth recently.
   **Strategic Implication:** Aggressive mid-season development can dramatically shift constructor rankings.
5. **Finding:** Market share of points per season visualizes clear "eras" of dominance.
   **Strategic Implication:** Teams must recognize era shifts and decide whether to invest in current cars or pivot to new regulations.


## Section 6 — Conclusion
Module 2 has revealed the fundamental truths of team dominance, reliability, and operational efficiency in Formula 1. By combining these insights with the driver analytics from Module 1, we now have a comprehensive understanding of the foundational performance blocks.

**Coming up next — Module 3: Race Strategy & Pit Stop Analytics**, where we will dive into tire degradation, undercuts, and race-day decision making!
